In [ ]:
# ===== APPROACH B — CELL 0 : setup =====
# কাজ একটাই: image+text জোড়ার জন্য cross-encoder relevance score তৈরি করা।
# Feature engineering বা training এখানে নেই — সেটা Approach A করবে।
# A নিজে থেকেই rr_train.npy / rr_test.npy খুঁজে নেয়, তাই দুটো account স্বাধীনভাবে চলতে পারে।
#
# Accelerator: GPU।  Internet: ON।  Input: essentials, img384
import os, glob, json, time, gc, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

OUT, INP = '/kaggle/working', '/kaggle/input'
T0 = time.time()
def find(n, isdir=False):
    for r in (INP, OUT):
        for h in glob.glob(f'{r}/**/{n}', recursive=True):
            if os.path.isdir(h) == isdir: return h
    return None
def tlog(*a): print(f'[{(time.time()-T0)/60:5.1f} min]', *a, flush=True)

import torch
VRAM = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
tlog('VRAM', f'{VRAM:.0f}GB')

ESS  = os.path.dirname(find('meta_train.parquet'))
IMGD = find('img384', isdir=True)
mtr = pd.read_parquet(f'{ESS}/meta_train.parquet')
mte = pd.read_parquet(f'{ESS}/meta_test.parquet')
for d in (mtr, mte):
    for c in ['t1','t2','h1','h2']: d[c] = d[c].astype(str)
    d['combo'] = np.where(d.t1 < d.t2, d.t1+'+'+d.t2, d.t2+'+'+d.t1)
mtr['y'] = mtr['label'].astype(str)
texts = pd.read_parquet(f'{ESS}/texts.parquet'); texts['hash'] = texts['hash'].astype(str)
TXTMAP = dict(zip(texts.hash, texts.text.fillna('').astype(str)))

def pairs(m):
    """image+text subset — caption আর figure আলাদা করে, মূল ক্রম বজায় রেখে"""
    d = m[m.combo == 'image+text'].reset_index(drop=True)
    sw = (d.t1 == 'image').values
    cap = [TXTMAP.get(h, '') for h in np.where(sw, d.h2, d.h1)]
    fig = list(np.where(sw, d.h1, d.h2))
    return d, cap, fig

dtr, cap_tr, fig_tr = pairs(mtr)
dte, cap_te, fig_te = pairs(mte)
tlog('image+text জোড়া — train', len(dtr), '| test', len(dte))


In [ ]:
# ===== APPROACH B — CELL 1 : Qwen3-VL-Reranker load + zero-shot যাচাই =====
# API: sentence-transformers CrossEncoder (model card-এর অফিশিয়াল পথ)
#   model.predict([(query_text, {"image": path}), ...], prompt=INSTR)
import subprocess, sys
subprocess.run(f'{sys.executable} -m pip install -q -U "sentence-transformers>=5" qwen-vl-utils',
               shell=True, timeout=1200)

MODEL = 'Qwen/Qwen3-VL-Reranker-2B'        # 8B এই সময়ে ধরবে না
tlog('model:', MODEL)

from sentence_transformers import CrossEncoder
model = CrossEncoder(MODEL, device='cuda', model_kwargs={'torch_dtype': torch.float16})
tlog('loaded')

INSTR = ('Given a caption from a scientific paper, retrieve the figure that comes '
         'from the same paper.')

def score(caps, figs, bs=8):
    """caption = query, figure = document। sigmoid দিয়ে 0..1-এ আনা।"""
    pairs = [(c[:1500], {'image': f'{IMGD}/{h}.jpg'}) for c, h in zip(caps, figs)]
    s = model.predict(pairs, prompt=INSTR, batch_size=bs, show_progress_bar=False,
                      activation_fn=torch.nn.Sigmoid())
    return np.asarray(s, dtype=np.float32).reshape(-1)

# ---- zero-shot যাচাই: ৩০০ জোড়া ----
n = 300
idx = np.random.default_rng(0).choice(len(dtr), n, replace=False)
t = time.time()
s0 = score([cap_tr[i] for i in idx], [fig_tr[i] for i in idx])
per = (time.time()-t)/n
g = pd.Series(s0).groupby(dtr.y.values[idx]).mean()
print('\nzero-shot score — class-ভিত্তিক গড়'); print(g.round(4).to_string())
eff = (g.max()-g.min())/max(s0.std(), 1e-9)
print(f'ব্যাপ্তি {g.max()-g.min():.4f} | effect {eff:.2f}')
print(f'প্রতি জোড়ায় {per:.2f}s → ৮০০০ জোড়ায় ≈ {per*8000/60:.0f} মিনিট')
print('\n→ effect > 0.3 হলে score কাজে লাগবে। এই cell কিছু থামায় না; cell 2 এমনিতেই চলবে।')


In [ ]:
# ===== APPROACH B — CELL 2 : পূর্ণ scoring → rr_train.npy / rr_test.npy =====
BUDGET = 150 * 60          # সময়সীমা; যতটুকু হয়েছে তাই সেভ হবে

def run(tag, caps, figs):
    ck = f'{OUT}/_ck_rr_{tag}.npy'
    done = list(np.load(ck)) if os.path.exists(ck) else []
    if done: tlog(f'{tag}: checkpoint থেকে {len(done)}')
    t = time.time()
    for i in range(len(done), len(caps), 256):
        done.extend(score(caps[i:i+256], figs[i:i+256]))
        np.save(ck, np.array(done, dtype=np.float32))
        el = time.time()-t; nd = max(len(done)-1, 1)
        tlog(f'  {tag} {len(done)}/{len(caps)}  বাকি ~{(len(caps)-len(done))*el/nd/60:.0f}m')
        if time.time()-t > BUDGET:
            tlog(f'⏱️ সময়সীমা — {tag} {len(done)}/{len(caps)} পর্যন্ত'); break
    a = np.array(done, dtype=np.float32)
    if len(a) < len(caps):                  # অসম্পূর্ণ হলে বাকিটা গড় দিয়ে ভরাট
        a = np.concatenate([a, np.full(len(caps)-len(a), float(a.mean()) if len(a) else 0.0, np.float32)])
    return a

rr_tr = run('tr', cap_tr, fig_tr)
rr_te = run('te', cap_te, fig_te)
np.save(f'{OUT}/rr_train.npy', rr_tr); np.save(f'{OUT}/rr_test.npy', rr_te)

g = pd.Series(rr_tr).groupby(dtr.y.values).mean()
print('\nপূর্ণ train score — class-ভিত্তিক গড়'); print(g.round(4).to_string())
eff = (g.max()-g.min())/max(rr_tr.std(), 1e-9)
print(f'ব্যাপ্তি {g.max()-g.min():.4f} | effect {eff:.2f}')
json.dump({'range': float(g.max()-g.min()), 'effect': float(eff),
           'by_class': {k: float(v) for k, v in g.items()}},
          open(f'{OUT}/report_B.json','w'), indent=1)
tlog('saved → rr_train.npy, rr_test.npy')
print('\n👉 Quick Save → Output কে Dataset বানিয়ে Approach A-তে attach করো।')
print('   A-র cell 3 নিজে থেকেই খুঁজে নিয়ে feature-এ যোগ করবে।')
